# Cross-Validation and Hyperparameter Tuning

In this notebook, we summarize our analysis of cross-validation and hyperparameter tuning for the kNN Regressor model. We also demonstrate the implementation of the Python script `kNN_script.py`.

Suppose we are interested in evaluating the cross-validation error scores for a particular prediction window. Our method performs cross-validation in a rolling fashion: for each day in the prediction window, we set the training window to the prior 60 days. We then obtain the predicted value for that day, move to the next day, and repeat the process by shifting the training window accordingly. At the end, we return all predicted values for the chosen hyperparameter combination.

## Documentation of `kNN_script.py`
We have developed a Python script named `kNN_script.py` that allows the user to specify hyperparameter combinations, the training window length, and the start and duration of the prediction window. The script defines a class `kNN_Cross_Validation` with the following main features:

- The constructor accepts the arguments `pca_comps: List[int]`, `n_nbrs: List[int]`, `days_to_train_on: int`, `mode: str`, `feature_selection: str`, `with_confidence_interval: bool`, and `no_iters: int`. 
- By default, the best-performing hyperparameters are set:`pca_comps = [35]` and `n_nbrs = [5]`. 
- The training window length can be customized using the `days_to_train_on` parameter (default: 60 days).
- The argument `mode` determines which dataframe to use:
    - `Validation` runs on data from 2019 to 2022 (default setting).
    - `Testing` runs on data from 2019 to 2023.
- The argument `feature_selection` determines which feature to use:
    - `spd_cubed_div_temp` runs on the feature (wind speed)$^3$ divided by the temperature in Kelvin at each farm (default setting).
    - `speed_cubed` runs on the feature (wind speed)$^3$ at each farm.
    - `speed` runs on the feature wind speed at each farm.
- The boolean parameter `with_confidence_interval` determines if we want to calculate confidence interval for different estimators. When this is set to `True`:
    - For each day in the testing window, we resample half of the training datapoints with replacement, i.e. bootstrapping (e.g. if `days_to_train_on` equals 60, we have $30\times 24$ datapoints choosen from the prior 60 day window for the testing day) and run the model `no_iters` times.
    - `no_iters` determines the number of iterations on each day of the training window.
    - These confidence intervals are stored in the attributes `pred_conf_intervals`, `mape_conf_intervals`, `r2_conf_intervals`, and `mae_conf_intervals`.
    


#### Key Methods:
- `input()`: Prompts the user to provide the validation or testing window.
- `manual_input()`: Lets the user to input validation or testing window (same functionality as `input()` without interactive prompts).
- `run()`: loads data, validates input, runs kNN forecasting, and computes evaluation metrics. Accepts a boolean argument `display` that determines if the plots are automatically displayed.
#### Behaviour:
- If only a single `(pca_comp, n_nbr)` pair was given to the constructor (default setting), it runs the model with that configuration and returns predictions, scores, and a `plotly.graph_objects.Figure` object. The plot is automatically displayed (by default).
- If multiple hyperparameter combinations are provided, it performs a manual grid search, runs predictions for each combination, and displays plots (by dafault) for the best-performing models based on MAPE, R² and MAE.
#### Modeling Approach:
- The kNN model is trained on the engineered feature: (wind speed)$^3$ divided by temperature (in Kelvin) for each wind farm.
- We train on hourly data, predict on hourly data, and then add the predicted values for the 24 hours to get the prediction on a particular day.
- Each prediction is based on a rolling 60-day training window.

##### ⚠️ NOTE: 
This script does not handle real-time forecasting. The prediction window must fall strictly within `2019-03-02` (reserving at least 60 days for training) to `2023-12-31` (end of the validation/testing dataset).


## Example of execution of the script and error scores:

In [1]:
from kNN_script import kNN_Cross_Validation

#### Instantiate a `kNN_Cross_Validation` class object with default constructor.

In [6]:
cv = kNN_Cross_Validation()
cv.input()
cv.run()


Please enter forecast start date in the format YYYY-MM-DD.The date should be in between 2019-03-02 and 2022-12-31.

You have entered: 2022-01-01.

Please enter forecast window (in days). It should be an integer between 1 or 365.

Your forecast window is days between: 2022-01-01 and 2022-06-30

Cross validation running on your prediction window...



#### Instantiate a `kNN_Cross_Validation` object with `confidence interval calculations`
To reduce runtime, we only allow to run confidence interval calculation with single hyperparameters pair. 

In [8]:
knn_ci = kNN_Cross_Validation(with_confidence_interval=True)
knn_ci.manual_input('2022-01-01',181)
knn_ci.run()


Your forecast window is days between: 2022-01-01 and 2022-06-30

Cross validation running on your prediction window...

Note: Running with iterations... This may take longer time.
Error score: mean MAPE = 0.153, mean R2= 0.887, mean MAE = 4037.1611491712706.
Confidence interval for MAPE: (np.float64(0.14650698151776717), np.float64(0.15924484116401047)).
Confidence interval for R2: (np.float64(0.8805744829120833), np.float64(0.8959939535630497)).
Confidence interval for MAE: (np.float64(3910.5789502762427), np.float64(4194.874917127072)).


#### Instantiate a `kNN_Cross_Validation` class object with customized `hyperparameters`:

In [4]:
cv_multiple = kNN_Cross_Validation([10, 20, 25, 30, 35], [5, 10, 20, 30, 50], days_to_train_on=100)
cv_multiple.input()


Please enter forecast start date in the format YYYY-MM-DD.The date should be in between 2019-04-11 and 2022-12-31.

You have entered: 2022-01-01.

Please enter forecast window (in days). It should be an integer between 1 or 365.

Your forecast window is days between: 2022-01-01 and 2022-06-30



In [5]:
cv_multiple.run()

Cross validation running on your prediction window...

Prediction done with: PCA comps = 10, knn nbr = 5. Error scores: cv_mape = 0.161, cv_R2 = 0.890, MAE = 4006.6.
Prediction done with: PCA comps = 10, knn nbr = 10. Error scores: cv_mape = 0.156, cv_R2 = 0.892, MAE = 3943.4.
Prediction done with: PCA comps = 10, knn nbr = 20. Error scores: cv_mape = 0.153, cv_R2 = 0.895, MAE = 3903.5.
Prediction done with: PCA comps = 10, knn nbr = 30. Error scores: cv_mape = 0.151, cv_R2 = 0.895, MAE = 3908.4.
Prediction done with: PCA comps = 10, knn nbr = 50. Error scores: cv_mape = 0.151, cv_R2 = 0.893, MAE = 3953.8.
Prediction done with: PCA comps = 20, knn nbr = 5. Error scores: cv_mape = 0.152, cv_R2 = 0.897, MAE = 3902.7.
Prediction done with: PCA comps = 20, knn nbr = 10. Error scores: cv_mape = 0.145, cv_R2 = 0.901, MAE = 3787.6.
Prediction done with: PCA comps = 20, knn nbr = 20. Error scores: cv_mape = 0.144, cv_R2 = 0.899, MAE = 3828.5.
Prediction done with: PCA comps = 20, knn nbr = 30.

#### Instantiate a `kNN_Cross_Validation` object with customized `feature selection`.

In [7]:
knn_feature = kNN_Cross_Validation([10, 20, 25, 30, 35], [5, 10, 20, 30, 50],feature_selection="speed")
knn_feature.manual_input('2022-01-01', 181)
knn_feature.run()


Your forecast window is days between: 2022-01-01 and 2022-06-30

Cross validation running on your prediction window...

Prediction done with: PCA comps = 10, knn nbr = 5. Error scores: cv_mape = 0.218, cv_R2 = 0.841, MAE = 4990.7.
Prediction done with: PCA comps = 10, knn nbr = 10. Error scores: cv_mape = 0.229, cv_R2 = 0.834, MAE = 5115.8.
Prediction done with: PCA comps = 10, knn nbr = 20. Error scores: cv_mape = 0.243, cv_R2 = 0.819, MAE = 5356.6.
Prediction done with: PCA comps = 10, knn nbr = 30. Error scores: cv_mape = 0.252, cv_R2 = 0.812, MAE = 5473.5.
Prediction done with: PCA comps = 10, knn nbr = 50. Error scores: cv_mape = 0.267, cv_R2 = 0.796, MAE = 5667.7.
Prediction done with: PCA comps = 20, knn nbr = 5. Error scores: cv_mape = 0.230, cv_R2 = 0.831, MAE = 5137.6.
Prediction done with: PCA comps = 20, knn nbr = 10. Error scores: cv_mape = 0.244, cv_R2 = 0.820, MAE = 5354.3.
Prediction done with: PCA comps = 20, knn nbr = 20. Error scores: cv_mape = 0.254, cv_R2 = 0.812,

#### Running with small testing window to see the confidence interval clearly

In [9]:
knn_ci = kNN_Cross_Validation(with_confidence_interval=True, days_to_train_on= 180)
knn_ci.manual_input('2022-01-01', 15)
knn_ci.run()


Your forecast window is days between: 2022-01-01 and 2022-01-15

Cross validation running on your prediction window...

Note: Running with iterations... This may take longer time.
Error score: mean MAPE = 0.157, mean R2= 0.936, mean MAE = 3517.9832.
Confidence interval for MAPE: (np.float64(0.1311467231617165), np.float64(0.18737408978337203)).
Confidence interval for R2: (np.float64(0.9109655006997743), np.float64(0.9522008135594904)).
Confidence interval for MAE: (np.float64(2969.124), np.float64(4193.798000000002)).


## Tries after I modified the method
Shifting wind power by 24 hours, so that we don't use any weather information about the prediction day (no forecasting). In this case, we train with the pairs $\{(X_n,y_{n+24})\}$ where $X_n$ is the weather on at hour $n$ against the target $y_{n+24}$ at hour $n+24$.

In [ ]:
from kNN_script import kNN_Cross_Validation

for no_days in [365, 546, 730]:
    for feature in ["spd_cubed_div_temp", "speed", "speed_cubed"]:
        knn = kNN_Cross_Validation([10, 20, 25, 30], [5, 10, 25, 50], 
                                   days_to_train_on= no_days, feature_selection=feature)
        knn.manual_input('2022-01-01', 181)
        knn.run()


Your forecast window is days between: 2022-01-01 and 2022-06-30

Cross validation running on your prediction window...

Prediction done with: PCA comps = 10, knn nbr = 5. Error scores: cv_mape = 0.446, cv_R2 = 0.333, MAE = 10295.1.
Prediction done with: PCA comps = 10, knn nbr = 10. Error scores: cv_mape = 0.440, cv_R2 = 0.357, MAE = 10151.9.
Prediction done with: PCA comps = 10, knn nbr = 25. Error scores: cv_mape = 0.435, cv_R2 = 0.372, MAE = 10079.4.
Prediction done with: PCA comps = 10, knn nbr = 50. Error scores: cv_mape = 0.436, cv_R2 = 0.364, MAE = 10124.6.
Prediction done with: PCA comps = 20, knn nbr = 5. Error scores: cv_mape = 0.438, cv_R2 = 0.316, MAE = 10349.3.
Prediction done with: PCA comps = 20, knn nbr = 10. Error scores: cv_mape = 0.436, cv_R2 = 0.336, MAE = 10277.0.
Prediction done with: PCA comps = 20, knn nbr = 25. Error scores: cv_mape = 0.431, cv_R2 = 0.358, MAE = 10121.2.
Prediction done with: PCA comps = 20, knn nbr = 50. Error scores: cv_mape = 0.429, cv_R2 =


Your forecast window is days between: 2022-01-01 and 2022-06-30

Cross validation running on your prediction window...

Prediction done with: PCA comps = 10, knn nbr = 5. Error scores: cv_mape = 0.467, cv_R2 = 0.257, MAE = 10397.1.
Prediction done with: PCA comps = 10, knn nbr = 10. Error scores: cv_mape = 0.459, cv_R2 = 0.299, MAE = 10231.8.
Prediction done with: PCA comps = 10, knn nbr = 25. Error scores: cv_mape = 0.454, cv_R2 = 0.333, MAE = 10019.0.
Prediction done with: PCA comps = 10, knn nbr = 50. Error scores: cv_mape = 0.457, cv_R2 = 0.343, MAE = 10079.4.
Prediction done with: PCA comps = 20, knn nbr = 5. Error scores: cv_mape = 0.470, cv_R2 = 0.229, MAE = 10631.1.
Prediction done with: PCA comps = 20, knn nbr = 10. Error scores: cv_mape = 0.456, cv_R2 = 0.273, MAE = 10375.6.
Prediction done with: PCA comps = 20, knn nbr = 25. Error scores: cv_mape = 0.446, cv_R2 = 0.329, MAE = 10045.4.
Prediction done with: PCA comps = 20, knn nbr = 50. Error scores: cv_mape = 0.446, cv_R2 =


Your forecast window is days between: 2022-01-01 and 2022-06-30

Cross validation running on your prediction window...

Prediction done with: PCA comps = 10, knn nbr = 5. Error scores: cv_mape = 0.447, cv_R2 = 0.321, MAE = 10427.4.
Prediction done with: PCA comps = 10, knn nbr = 10. Error scores: cv_mape = 0.438, cv_R2 = 0.346, MAE = 10209.4.
Prediction done with: PCA comps = 10, knn nbr = 25. Error scores: cv_mape = 0.436, cv_R2 = 0.354, MAE = 10182.8.
Prediction done with: PCA comps = 10, knn nbr = 50. Error scores: cv_mape = 0.438, cv_R2 = 0.354, MAE = 10188.5.
Prediction done with: PCA comps = 20, knn nbr = 5. Error scores: cv_mape = 0.439, cv_R2 = 0.307, MAE = 10426.2.
Prediction done with: PCA comps = 20, knn nbr = 10. Error scores: cv_mape = 0.438, cv_R2 = 0.320, MAE = 10353.0.
Prediction done with: PCA comps = 20, knn nbr = 25. Error scores: cv_mape = 0.435, cv_R2 = 0.341, MAE = 10223.6.
Prediction done with: PCA comps = 20, knn nbr = 50. Error scores: cv_mape = 0.434, cv_R2 =


Your forecast window is days between: 2022-01-01 and 2022-06-30

Cross validation running on your prediction window...

Prediction done with: PCA comps = 10, knn nbr = 5. Error scores: cv_mape = 0.447, cv_R2 = 0.342, MAE = 10145.7.
Prediction done with: PCA comps = 10, knn nbr = 10. Error scores: cv_mape = 0.439, cv_R2 = 0.367, MAE = 9947.1.
Prediction done with: PCA comps = 10, knn nbr = 25. Error scores: cv_mape = 0.433, cv_R2 = 0.397, MAE = 9819.3.
Prediction done with: PCA comps = 10, knn nbr = 50. Error scores: cv_mape = 0.429, cv_R2 = 0.409, MAE = 9772.1.
Prediction done with: PCA comps = 20, knn nbr = 5. Error scores: cv_mape = 0.436, cv_R2 = 0.328, MAE = 10242.3.
Prediction done with: PCA comps = 20, knn nbr = 10. Error scores: cv_mape = 0.434, cv_R2 = 0.343, MAE = 10130.3.
Prediction done with: PCA comps = 20, knn nbr = 25. Error scores: cv_mape = 0.425, cv_R2 = 0.385, MAE = 9904.8.
Prediction done with: PCA comps = 20, knn nbr = 50. Error scores: cv_mape = 0.424, cv_R2 = 0.4


Your forecast window is days between: 2022-01-01 and 2022-06-30

Cross validation running on your prediction window...

Prediction done with: PCA comps = 10, knn nbr = 5. Error scores: cv_mape = 0.460, cv_R2 = 0.221, MAE = 10635.7.
Prediction done with: PCA comps = 10, knn nbr = 10. Error scores: cv_mape = 0.457, cv_R2 = 0.251, MAE = 10538.2.
Prediction done with: PCA comps = 10, knn nbr = 25. Error scores: cv_mape = 0.454, cv_R2 = 0.298, MAE = 10356.5.
Incorrect dataframe passed to extract_train_test_data.
Error in hyperparameters combination: (10, 50)


In [4]:
from kNN_script import kNN_Cross_Validation
knn = kNN_Cross_Validation(days_to_train_on=360)
knn.manual_input("2022-01-01", 181)
knn.run()


Your forecast window is days between: 2022-01-01 and 2022-06-30

Cross validation running on your prediction window...



In [2]:
import pandas as pd

# Create a sample DataFrame
data = {'Name': ['Alice', 'Bob', 'Charlie', 'David'],
        'Score': [85, 92, 78, 95],
        'Grade': ['A', 'A', 'B', 'A']}
df = pd.DataFrame(data)

print("Original DataFrame:")
print(df)

# Shift the 'Score' column downwards by 1 period
df['Score'] = df['Score'].shift(periods=1)

print("\nDataFrame after shifting 'Score' column downwards:")
print(df)

# Shift the 'Grade' column upwards by 2 periods
df['Grade'] = df['Grade'].shift(periods=-2)

print("\nDataFrame after shifting 'Grade' column upwards:")
print(df)

Original DataFrame:
      Name  Score Grade
0    Alice     85     A
1      Bob     92     A
2  Charlie     78     B
3    David     95     A

DataFrame after shifting 'Score' column downwards:
      Name  Score Grade
0    Alice    NaN     A
1      Bob   85.0     A
2  Charlie   92.0     B
3    David   78.0     A

DataFrame after shifting 'Grade' column upwards:
      Name  Score Grade
0    Alice    NaN     B
1      Bob   85.0     A
2  Charlie   92.0  None
3    David   78.0  None
